# Hydrologic Flow (Zion) — Student Notebook
**Course:** Earth Data Viz (sophomore majors)

### What you’ll do
1. Download **daily discharge** data for two USGS gages near Zion.
2. Clean + convert units (cfs → m³/s).
3. Compute summary statistics and export a **Table 1** to Excel.
4. Create a hydrograph figure and export it as a PNG for Word.

✅ Keep an eye out for **TODO** cells — those are the parts you complete.


In [ ]:
# ===== Setup =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import StringIO

pd.set_option('display.max_columns', 50)


## 1) Choose your analysis window
Use the **1-year period ending on the field trip**.

**TODO:** Confirm these dates match *your* trip year.


In [ ]:
# TODO: update if your field trip year differs
start_date = '2025-03-07'
end_date   = '2026-03-07'

field_trip_start = '2026-03-05'
field_trip_end   = '2026-03-07'


## 2) Define Zion-area gages
We’ll compare the **North Fork Virgin River** and **East Fork Virgin River** near Springdale.

Optional: add a downstream site at Virgin, UT.


In [ ]:
sites = {
    'North Fork Virgin (Springdale)': '09405500',
    'East Fork Virgin (Springdale)':  '09404900',
    # Optional:
    # 'Virgin River (Virgin, UT)':      '09406000',
}
sites

## 3) Download daily discharge (USGS Water Services)
We’ll use the **Daily Values (dv)** service with parameter code **00060** (discharge).

### Your task
Complete the TODOs in the function so it returns a clean dataframe with:
- `date`
- `discharge_cfs`
- `site_name`
- `site_no`


In [ ]:
def fetch_daily_discharge(site_no: str, start: str, end: str) -> pd.DataFrame:
    """Fetch USGS daily mean discharge (cfs) for one site between start/end dates.
    Returns a tidy dataframe with one row per day.
    """
    base = 'https://waterservices.usgs.gov/nwis/dv/'
    params = {
        'format': 'json',
        'sites': site_no,
        'startDT': start,
        'endDT': end,
        'parameterCd': '00060',
        # TODO: ensure we are pulling daily MEAN discharge
        'statCd': '00003',
    }

    # TODO: make the web request and raise an error if it fails
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()

    # TODO: pull out the time series values
    ts = j['value']['timeSeries'][0]
    site_name = ts['sourceInfo']['siteName']
    values = ts['values'][0]['value']  # list of dicts

    df = pd.DataFrame(values)
    # expected columns: 'value', 'dateTime', 'qualifiers'...
    df = df.rename(columns={'value': 'discharge_cfs', 'dateTime': 'dateTime'})

    # TODO: convert strings → numeric/datetime and keep only what we need
    df['discharge_cfs'] = pd.to_numeric(df['discharge_cfs'], errors='coerce')
    df['date'] = pd.to_datetime(df['dateTime']).dt.date
    df = df[['date', 'discharge_cfs']].copy()
    df['site_name'] = site_name
    df['site_no'] = site_no

    return df


### Download for all sites
**TODO:** Use a loop (or list comprehension) to download for each gage and combine them.


In [ ]:
# TODO: download all sites and combine into one dataframe called discharge_df
frames = []
for label, site_no in sites.items():
    tmp = fetch_daily_discharge(site_no, start_date, end_date)
    tmp['label'] = label
    frames.append(tmp)

discharge_df = pd.concat(frames, ignore_index=True)

discharge_df.head()

## 4) Clean + convert units
**TODO:**
1. Drop missing discharge values.
2. Convert cfs → m³/s.
3. Make sure `date` is a datetime column (not python `date`).


In [ ]:
# TODO: clean + convert
discharge_df = discharge_df.dropna(subset=['discharge_cfs']).copy()
discharge_df['date'] = pd.to_datetime(discharge_df['date'])

# Conversion: 1 cubic foot = 0.028316846592 cubic meters
discharge_df['discharge_m3s'] = discharge_df['discharge_cfs'] * 0.028316846592

discharge_df[['date','label','discharge_cfs','discharge_m3s']].head()

## 5) Compute statistics (Table 1)
Compute (in **m³/s**): min, max, mean, median, standard deviation.

For “total discharge,” compute **annual volume** from daily mean discharge:
- volume (m³) = discharge (m³/s) × 86,400 s/day, summed across days.


In [ ]:
# TODO: compute stats by site
stats = (discharge_df
         .groupby(['label','site_no'], as_index=False)
         .agg(min_discharge_m3s=('discharge_m3s','min'),
              max_discharge_m3s=('discharge_m3s','max'),
              mean_discharge_m3s=('discharge_m3s','mean'),
              median_discharge_m3s=('discharge_m3s','median'),
              std_discharge_m3s=('discharge_m3s','std'),
              days=('discharge_m3s','size')))

# annual volume (m³)
sec_per_day = 86400
vol = (discharge_df.assign(volume_m3 = discharge_df['discharge_m3s'] * sec_per_day)
       .groupby(['label','site_no'], as_index=False)
       .agg(total_volume_m3=('volume_m3','sum')))

table1 = stats.merge(vol, on=['label','site_no'])

# Round for reporting (hundredths for m3/s; whole m3 for volume)
for c in [c for c in table1.columns if c.endswith('_m3s')]:
    table1[c] = table1[c].round(2)
table1['total_volume_m3'] = table1['total_volume_m3'].round(0)

table1

## 6) Export deliverables
Export:
- Cleaned daily time series (Excel)
- Table 1 summary stats (Excel)
- Gage locations for ArcGIS (CSV)


In [ ]:
# Cleaned time series
discharge_df.to_excel('zionsites_daily_discharge_clean.xlsx', index=False)

# Table 1
table1.to_excel('table1_hydrologic_flow_zion.xlsx', index=False)

print('Wrote: zionsites_daily_discharge_clean.xlsx')
print('Wrote: table1_hydrologic_flow_zion.xlsx')


### (Optional) Fetch site coordinates for ArcGIS
This uses the USGS **site** service (tab-delimited “RDB” format).


In [ ]:
def fetch_site_locations(site_nos):
    base = 'https://waterservices.usgs.gov/nwis/site/'
    params = {
        'format': 'rdb',
        'sites': ','.join(site_nos),
    }
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    txt = r.text
    df = pd.read_csv(StringIO(txt), sep='\t', comment='#', dtype=str)
    keep = ['site_no','station_nm','dec_lat_va','dec_long_va']
    df = df[keep].copy()
    df['dec_lat_va'] = pd.to_numeric(df['dec_lat_va'], errors='coerce')
    df['dec_long_va'] = pd.to_numeric(df['dec_long_va'], errors='coerce')
    return df

loc_df = fetch_site_locations(list(sites.values()))
loc_df.to_csv('zionsites_usgs_gages_for_arcgis.csv', index=False)
loc_df

## 7) Create Figure 1 (Hydrograph)
Requirements:
- Both sites on the same axes
- Axis labels and units
- Legend
- No title
- Mark the **field trip window** (Mar 5–7) on the plot


In [ ]:
# Pivot wide for plotting
wide = discharge_df.pivot_table(index='date', columns='label', values='discharge_m3s', aggfunc='mean').sort_index()

fig, ax = plt.subplots(figsize=(9,4))
wide.plot(ax=ax)

# Field trip window shading
fts = pd.to_datetime(field_trip_start)
fte = pd.to_datetime(field_trip_end)
ax.axvspan(fts, fte, alpha=0.2)

ax.set_xlabel('Date')
ax.set_ylabel('Discharge (m³/s)')
ax.legend(loc='upper left', frameon=False)
ax.grid(False)

fig.tight_layout()
fig.savefig('figure1_hydrograph_zion.png', dpi=300)
plt.show()

print('Saved: figure1_hydrograph_zion.png')


## 8) Field observation annotation (required)
During the field trip, record one observation at the Virgin River (time + notes).

**TODO:** Replace the placeholder values below, then re-run the plot cell to add an annotation.


In [ ]:
field_obs_time = '2026-03-06 10:30'  # TODO
field_obs_note = 'Observed moderate flow; slightly turbid; overcast.'  # TODO


In [ ]:
# Add a point annotation at the nearest day
obs_dt = pd.to_datetime(field_obs_time)
nearest_day = pd.to_datetime(obs_dt.date())

fig, ax = plt.subplots(figsize=(9,4))
wide.plot(ax=ax)

ax.axvspan(pd.to_datetime(field_trip_start), pd.to_datetime(field_trip_end), alpha=0.2)

# Annotate at North Fork value (if available). If missing, annotate at first series.
series_name = wide.columns[0]
yval = wide.loc[nearest_day, series_name] if nearest_day in wide.index else np.nan
if np.isfinite(yval):
    ax.scatter([nearest_day], [yval])
    ax.annotate(field_obs_note, xy=(nearest_day, yval), xytext=(10, 10), textcoords='offset points')

ax.set_xlabel('Date')
ax.set_ylabel('Discharge (m³/s)')
ax.legend(loc='upper left', frameon=False)
ax.grid(False)
fig.tight_layout()
plt.show()
